# Eksploracja ekstrakcji — analiza per rozdział

Łączy techniki ze wszystkich poprzednich notebooków:
- **part2**: ręczny parser TOC (regex ze strony spisu treści)
- **part3**: obliczenie content bbox (bez panelu nawigacyjnego)
- **explore_extraction_new**: `pymupdf4llm.to_json()` + wizualizacja boxów

Przepływ: tytuł → TOC → rozdziały → pomiń 1. stronę → pymupdf4llm → wizualizacja

In [ ]:
import fitz
import pymupdf.layout           # musi być przed pymupdf4llm
import pymupdf4llm
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

PDF_PATH = Path("../data/raport_2024_pl 1.pdf")
doc = fitz.open(PDF_PATH)
print(f"Stron: {len(doc)}")

## Krok 1a — Tytuł dokumentu

In [ ]:
# Próba 1: metadane PDF
doc_title = doc.metadata.get("title", "").strip()

# Fallback: największy span na stronie 0 (TideSans-600Bunny, size >= 40)
if not doc_title:
    for block in doc[0].get_text("dict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                if round(span["size"]) >= 40:
                    doc_title = span["text"].strip()
                    break

print(f"Tytuł dokumentu: {doc_title!r}")

## Krok 1b — Spis treści (parser regex ze strony TOC)

In [ ]:
# Wzorce z explore_extraction_part2.ipynb
PATTERN_CHAPTER_NUM  = re.compile(r'^[IVXLCDM]+\s*$')
PATTERN_CHAPTER_FULL = re.compile(r'^([IVXLCDM]+)\s+(.+?)\s*$')
PATTERN_PAGE_ONLY    = re.compile(r'^\d+\s*$')
PATTERN_SUB_FULL     = re.compile(r'^\d+\.\s+(.+?)\s*\.{2,}\s*(\d+)\s*$')
PATTERN_SUB_START    = re.compile(r'^\d+\.\s+(.+)$')
RE_DOTS_END          = re.compile(r'^(.*?)\s*\.{2,}\s*(\d+)\s*$')

toc_entries = []  # list of (title, page_1based, level)
toc_page = doc[1]  # strona spisu treści (0-based indeks 1)
data = toc_page.get_text("dict")

for block in data["blocks"]:
    if block["type"] != 0:
        continue
    lines = [
        "".join(s["text"] for s in line["spans"]).strip()
        for line in block["lines"]
    ]
    lines = [l for l in lines if l]
    i = 0
    while i < len(lines):
        line = lines[i]

        # 3-liniowy nagłówek: "I" / "Jedyny taki bank" / "3"
        if (PATTERN_CHAPTER_NUM.match(line)
                and i + 2 < len(lines)
                and PATTERN_PAGE_ONLY.match(lines[i + 2])):
            toc_entries.append((f"{line.strip()} {lines[i+1].strip()}", int(lines[i+2].strip()), 1))
            i += 3
            continue

        # 2-liniowy nagłówek: "VIII Perspektywy" / "142"
        mc = PATTERN_CHAPTER_FULL.match(line)
        if mc and i + 1 < len(lines) and PATTERN_PAGE_ONLY.match(lines[i + 1]):
            toc_entries.append((f"{mc.group(1)} {mc.group(2).strip()}", int(lines[i+1].strip()), 1))
            i += 2
            continue

        # Podpunkt pełny: "1. Tytuł ..... 42"
        ms = PATTERN_SUB_FULL.match(line)
        if ms:
            toc_entries.append((ms.group(1).strip(), int(ms.group(2)), 2))
            i += 1
            continue

        # Podpunkt zawijany: dwie linie
        ms2 = PATTERN_SUB_START.match(line)
        if ms2 and i + 1 < len(lines):
            m2 = RE_DOTS_END.match(lines[i + 1])
            if m2:
                title_full = f"{ms2.group(1).strip()} {m2.group(1).strip()}".strip()
                toc_entries.append((title_full, int(m2.group(2)), 2))
                i += 2
                continue
        i += 1

print(f"Wpisy TOC: {len(toc_entries)}\n")
for title, pg, lvl in toc_entries:
    indent = "  " * (lvl - 1)
    print(f"{indent}str.{pg:3d}  {title}")

## Krok 1c — Zakresy stron rozdziałów

In [ ]:
chapters = [(title, page) for title, page, level in toc_entries if level == 1]

chapter_ranges = []
for i, (title, start_page) in enumerate(chapters):
    is_last = (i + 1 == len(chapters))
    end_page = len(doc) if is_last else chapters[i + 1][1] - 1
    chapter_ranges.append({
        "title":         title,
        "start_page":    start_page,       # 1-based, strona okładkowa rozdziału
        "end_page":      end_page,         # 1-based, włącznie
        "content_start": start_page + 1,   # pomijamy 1. stronę (okładka rozdziału)
    })

print(f"Rozdziały ({len(chapter_ranges)}):\n")
for i, ch in enumerate(chapter_ranges):
    has_content = ch["content_start"] <= ch["end_page"]
    content_str = f"treść: str.{ch['content_start']}–{ch['end_page']}" if has_content else "jednostronicowy"
    print(f"  {i+1:2d}. str.{ch['start_page']:3d}–{ch['end_page']:3d}  [{content_str}]  {ch['title']}")

## Krok 1d — Content bbox (z explore_extraction_part3)

In [ ]:
# Globalny bounding box treści — pomijamy panel nawigacyjny (x0 < 160 pt)
NAV_X1_THRESHOLD = 160
all_x0, all_x1, all_y0, all_y1 = [], [], [], []

for page in doc:
    for block in page.get_text("blocks"):
        x0, y0, x1, y1 = block[:4]
        if x0 < NAV_X1_THRESHOLD:
            continue
        all_x0.append(x0)
        all_x1.append(x1)
        all_y0.append(y0)
        all_y1.append(y1)

content_rect = fitz.Rect(min(all_x0), min(all_y0), max(all_x1), max(all_y1))
print(f"Content rect: {content_rect}")
print(f"  szerokość: {content_rect.width:.0f} pt")
print(f"  wysokość:  {content_rect.height:.0f} pt")

## Krok 2 — Funkcja analizy i wizualizacji rozdziału

In [ ]:
TYPE_COLORS = {
    "text":           "orange",
    "list-item":      "orange",
    "footnote":       "orange",
    "picture":        "deeppink",
    "table":          "blue",
    "table-fallback": "blue",
    "title":          "green",
    "section-header": "green",
    "page-header":    "lightgray",
    "page-footer":    "lightgray",
    "formula":        "purple",
}
DEFAULT_COLOR = "gray"
SCALE = 0.3


def analyze_chapter(chapter_idx: int):
    ch = chapter_ranges[chapter_idx]
    print(f"\n{'='*60}")
    print(f"Rozdział {chapter_idx + 1}: {ch['title']}")
    print(f"Strony treści: {ch['content_start']}–{ch['end_page']}")
    print(f"{'='*60}")

    if ch["content_start"] > ch["end_page"]:
        print("  (brak stron treści — rozdział jednostronicowy)")
        return

    # Indeksy 0-based stron treści (pomijamy pierwszą stronę rozdziału)
    page_indices = list(range(ch["content_start"] - 1, ch["end_page"]))

    # Analiza layoutu pymupdf4llm — bezpośrednio na oryginalnym doc
    json_data = json.loads(pymupdf4llm.to_json(doc, pages=page_indices))

    # Wizualizacja per strona
    for local_idx, page_data in enumerate(json_data["pages"]):
        boxes = page_data.get("boxes", [])

        # Filtruj boxy do content_rect (wyklucz panel nawigacyjny i marginesy)
        boxes_in_content = [
            b for b in boxes
            if b["x1"] > content_rect.x0 and b["x0"] < content_rect.x1
            and b["y1"] > content_rect.y0 and b["y0"] < content_rect.y1
        ]

        orig_page_num = ch["content_start"] + local_idx
        page = doc[page_indices[local_idx]]

        # Render strony do obrazu numpy
        mat = fitz.Matrix(SCALE, SCALE)
        pix = page.get_pixmap(matrix=mat)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
            pix.height, pix.width, pix.n
        )
        if pix.n == 4:
            img = img[:, :, :3]

        fig, ax = plt.subplots(figsize=(12, 14))
        ax.imshow(img)
        ax.set_title(
            f"{ch['title']}  |  str. {orig_page_num}  |  {len(boxes_in_content)} bloków",
            fontsize=11,
        )

        # Przerywana ramka content_rect
        frame = patches.Rectangle(
            (content_rect.x0 * SCALE, content_rect.y0 * SCALE),
            content_rect.width * SCALE,
            content_rect.height * SCALE,
            linewidth=2, edgecolor="black", facecolor="none", linestyle="--",
        )
        ax.add_patch(frame)

        # Rysuj boxy z numeracją
        for idx, box in enumerate(boxes_in_content):
            x0, y0, x1, y1 = box["x0"], box["y0"], box["x1"], box["y1"]
            btype = box.get("boxclass", "unknown")
            color = TYPE_COLORS.get(btype, DEFAULT_COLOR)

            rect = patches.Rectangle(
                (x0 * SCALE, y0 * SCALE),
                (x1 - x0) * SCALE,
                (y1 - y0) * SCALE,
                linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.25,
            )
            ax.add_patch(rect)
            ax.text(
                x0 * SCALE + 2, y0 * SCALE + 10,
                f"{idx + 1} ({btype})",
                color=color, fontsize=8, weight="bold",
            )

        ax.axis("off")
        plt.tight_layout()
        plt.show()

In [ ]:
# Analiza pierwszych 3 rozdziałów
for i in range(min(3, len(chapter_ranges))):
    analyze_chapter(i)